# N-gram language model

**Naive bayes asks:**  
given the class, how likely it is the word?
P(Word | class)

**N-gram asks:**  
 given the previous word(s) how likely is the next word?  
P(word_t | word_{t-1}, word_{t-2}, ...)

## Sentence boundary tokens

`<s>` represents the start of a sentence 
`/<s>` represents the end of a sentence 

For an N-gram model of order n, we usually add n-1 tokens

Bigram model:  
`<s>` I love NLP `</s>`

Trigram model:  
`<s> <s>` I love NLP `</s>`
why?  
Because the model needs enough previous context even at the beginning of the sentence

## Build tokenizer

tokenizer converts raw text into sequence of tokens that the language model can process.

**Lowercasing reduces vocabulary size.**  

without lowercasing, `Machine` and `machine` are treated as different words, unnecessarily increasing the vocabulary.  

**Tokenization defines what a "word" is.** 

The language model does not understand raw text. It only sees the sequence of tokens produced by the tokenizer. Different tokenization strategies produce different vocabularies and therefore different probability estimates.


In [16]:
corpus = [
    "I love  machine  learning",
    "I love deep learning",
    "I enjoy natural language processing",
    "Machine learning is fun",
    "Deep learning is powerful"
]
sentence = "I   love   machine learning ! !"

In [17]:
def tokenize_sentence(sentence):
    puncs = ".?!,"
    sentence = sentence.lower()
    for punc in puncs:
        sentence = sentence.replace(punc,'')
    tokenized = sentence.split()
    return tokenized


In [18]:
def tokenize_corpus(corpus: list[str]):
    tokenized = []
    for sentence in corpus:
        tokenized.append(tokenize_sentence(sentence))
    return tokenized 
    #  Or return [tokenize_sentence(sentence) for sentence in corpus]

In [29]:
tokenized_corpus = tokenize_corpus(corpus)

## Build-vocabulary 

vocabulary is the set of all unique tokens that appear in the corpus.

In our implementation, we use Python's `set` because it automatically prevents duplicates entries.  

In [20]:
def build_vocabulary (tokenized_corpus):
    vocab = set()

    for sentence in tokenized_corpus:
        for word in sentence:
            vocab.add(word)

    return vocab

### Vocabulary size vs corpus size

**Vocabulary size**: The number of unique words in the corpus.  

**Corpus size**: The number of word occurrence (including duplicates).  

**Example**:  

Corpus:
`["i", "love", "nlp"]
["i", "love", "python"]`

Vocabulary Size = 4
`{"i", "love", "nlp", "python"}`

Corpus Size = 6
`i, love, nlp, i, love, python`

## Unigram counts

The unigram language model estimates the probability of a word without considering any surrounding context. A ungram count stores how many times each unique word appears in the corpus.  

Vocabulary answers: "what words exist?"  
Unigram counts answer: "How many times each word appear?"

**In Naive Bayes:**  

$$
P(word | class) = \frac{\text{count(word in class)}}{\text{total words in class}}
$$

In unigram we simply remove the conditioning:  
$$
P(word) = \frac{\text{count(word)}}{\text{total words}}
$$

The denominator is the total number of word occurrence (corpus size), not the vocabulary size. 

Example:  
$$
P(learning) = \frac{\text{count(learning)}}{ \sum_{\text{all words}} \text{count(words)}}
$$

Ungram probabilities always sum to 1.  


In [ ]:
def unigrams_counts(tokenized_corpus):
    count  = {}
    for sentence in tokenized_corpus:
        for word in sentence:
            if word in count:
                count[word] +=1
            else:
                count[word] = 1
    return (count)

In [22]:
unigrams_counted = unigrams_counts(tokenized_corpus)
print(unigrams_counted)

{'i': 3, 'love': 2, 'machine': 2, 'learning': 4, 'deep': 2, 'enjoy': 1, 'natural': 1, 'language': 1, 'processing': 1, 'is': 2, 'fun': 1, 'powerful': 1}


In [23]:
def compute_unigram_probabilities(unigram_counts:dict):
    total_words = sum(unigram_counts.values())
    
    unigram_probabilities = {}
    
    for word, value in unigram_counts.items():
        unigram_probabilities[word] = value/ total_words
    
    return (unigram_probabilities)

"""
Complexity

for vocabulary size V
we perform: sum(unigram_counts.values()) 
which visits every unique word once.

So total complexity:
- Time: O(V)
- Space: O(V)
"""

'\nComplexity\n\nfor vocabulary size V\nwe perform: sum(unigram_counts.values()) \nwhich visits every unique word once.\n\nSo total complexity:\n- Time: O(V)\n- Space: O(V)\n'

## Bigrams

Formula:

$$
P ( \text{current word} | \text {previous word})
$$

The probability of current word given teh previous word = (the number of times they appear together) / the number of times the previous word appears as a context (as the first bigram word) 

$$
P(w_i | w_{i-1}) = \frac {count (w_{i-1},w_i) } {count(w_{i-1})}
$$

The denominator is not the total number of words anymore.  
In the unigram model we normalized over the entire corpus, while in the bigram mode we normalize within each context.  
i.e. Among all the times the previous word appeared, how often was it followed by the current word?

The core idea of statistical language model: **The probability of the next word depends on the context**

---

## N-grams function 

**Sliding window**: algorithm technique  used to process consecutive elements efficiently.  

For N-grams:
- window size = n.
- -step = 1.

Sentence:
i love machine learning

n = 2

Windows:

(i, love)  
(love, machine)  
(machine, learning)  

**Formula**:

$$
\text {Number of n-grams=L−n+1}
$$

where 
- L: length of sentence.
- $
n \leq {L}
$


In [ ]:
def generate_ngrams(tokenized_corpus, n : int ):
    
    if n <= 0:
        raise ValueError("n must be greater than 0") 

    n_gram = []

    for sentence in tokenized_corpus:
        L = len(sentence)
        
        for i in range (L - n + 1):        

            n_gram.append(tuple(sentence [i:n+i]))
    
    return (n_gram)

""" 
Complexity:
- Time O(N)
- Space: O(number of generated n-grams)
"""

' \nComplexity:\n- Time O(N)\n- Space: O (number of generated n-grams)\n'

In [81]:
generated_ngrams = generate_ngrams(tokenized_corpus,2)

In [ ]:
def count_ngrams(generated_ngrams):
    count = {}
    for n_gram in generated_ngrams:
        if n_gram in count: # same as "if n_gram in count.keys()" dictionaries check their keys by default.
            count[n_gram] +=1
        else:
            count[n_gram] = 1 
    
    return count

""" 
Complexity:
- Time O(M) where M is the number of generated n-grams
- Space: O(U) where U is the number of unique n-grams
"""


In [ ]:
print(count_ngrams(generated_ngrams))

{('i', 'love'): 2, ('love', 'machine'): 1, ('machine', 'learning'): 2, ('love', 'deep'): 1, ('deep', 'learning'): 2, ('i', 'enjoy'): 1, ('enjoy', 'natural'): 1, ('natural', 'language'): 1, ('language', 'processing'): 1, ('learning', 'is'): 2, ('is', 'fun'): 1, ('is', 'powerful'): 1}
